# halarc1407 | 2026-03-28
## Stroke Prediction Dataset — Machine Learning: Logistic Regression
**SDC380 | Week 4 Project**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              confusion_matrix, mean_absolute_error,
                              mean_squared_error, r2_score)
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

todays_date = '2026-03-28'
student_id = 'halarc1407'
display(Markdown(f'**Date:** {todays_date}  |  **Student ID:** {student_id}'))

In [ ]:
# Load dataset
df = pd.read_excel('Stroke_Prediction_Dataset.xlsx')
display(Markdown(f'**Shape:** {df.shape}'))
display(df.head())

In [ ]:
# Step 3: One-hot encode categorical columns
cat_cols = ['Gender', 'Marital Status', 'Work Type', 'Residence Type',
            'Smoking Status', 'Alcohol Intake', 'Physical Activity',
            'Family History of Stroke', 'Dietary Habits']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=False)
df_encoded['Stroke'] = (df_encoded['Diagnosis'] == 'Stroke').astype(int)
df_encoded = df_encoded.drop(columns=['Patient ID', 'Patient Name', 'Diagnosis'])
display(Markdown(f'**Encoded shape:** {df_encoded.shape}'))
display(Markdown('One-hot encoded columns: ' + str([c for c in df_encoded.columns if '_' in c][:12])))
display(df_encoded.head())

In [ ]:
# Step 4: Train/test split 80/20
X = df_encoded.drop('Stroke', axis=1)
y = df_encoded['Stroke']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
display(Markdown(f'**Training set:** {X_train.shape[0]} rows | **Test set:** {X_test.shape[0]} rows | **Features:** {X_train.shape[1]}'))

In [ ]:
# Step 5: Train Logistic Regression model
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)
display(Markdown(f'### {student_id} - {todays_date}'))
display(Markdown('**Model:** LogisticRegression(max_iter=1000, random_state=42)'))
display(Markdown(f'**Training samples:** {X_train.shape[0]} | **Features:** {X_train.shape[1]}'))

In [ ]:
# Step 6: Evaluate model - MAE, MSE, R2, and classification metrics
y_pred = lr.predict(X_test_s)
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
display(Markdown(f'### {student_id} - {todays_date}'))
display(Markdown(f'**Accuracy:** {acc:.4f} | **Precision:** {prec:.4f} | **Recall:** {rec:.4f}'))
display(Markdown(f'**MAE:** {mae:.4f} | **MSE:** {mse:.4f} | **R²:** {r2:.4f}'))
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Stroke','Stroke'], yticklabels=['No Stroke','Stroke'])
ax.set_title(f'Confusion Matrix - {student_id} - {todays_date}')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

In [ ]:
# Step 7: Feature importance via logistic regression coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr.coef_[0],
    'Abs_Coefficient': np.abs(lr.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)
display(Markdown(f'### {student_id} - {todays_date}'))
display(Markdown('**Top 15 Features by |Coefficient| (Feature Importance):**'))
display(coef_df.head(15))
top15 = coef_df.head(15)
colors = ['#C00000' if v > 0 else '#2E75B6' for v in top15['Coefficient']]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(top15)), top15['Abs_Coefficient'], color=colors)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['Feature'])
ax.invert_yaxis()
ax.set_xlabel('|Coefficient|')
ax.set_title(f'Feature Importance - {student_id} - {todays_date}')
plt.tight_layout(); plt.show()